### Importações para Rerank RAG

Bibliotecas novas em relação aos notebooks anteriores:

- **ContextualCompressionRetriever**: Retriever que aplica uma etapa de compressão/filtragem sobre os resultados de um retriever base
- **CohereRerank**: Modelo de reranking da Cohere que reordena documentos por relevância real — usa um modelo de linguagem dedicado à tarefa de ordenação, diferente de embeddings

A grande novidade deste notebook é a **estratégia em dois estágios**:
1. Recuperação ampla com retriever naive (busca por similaridade)
2. Reranking preciso com modelo Cohere (filtragem por relevância)

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

### Modelos de Embedding e LLM

- `text-embedding-3-small`: Mesmo modelo de embedding dos outros notebooks — converte texto em vetores para busca por similaridade
- `max_tokens=300`: Resposta mais curta que os outros notebooks (200 no RAG simples, 500 no parent, 1000 no code review) — adequado para respostas diretas sobre legislação

In [10]:
# CARREGAR MODELOS OPENAI - Embeddings e CHAT

embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model_name='gpt-3.5-turbo', max_tokens=300)

### Carregamento do Documento PDF

Mesmo fluxo dos notebooks anteriores:
- `PyPDFLoader` extrai o texto do PDF página por página
- `load_and_split()` retorna uma lista com 31 documentos (um por página)
- Esses documentos serão divididos em chunks menores na próxima etapa

In [11]:
pdf_link = './assets/anexo-projeto-de-lei.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

### Divisão em Chunks

`RecursiveCharacterTextSplitter` com os mesmos parâmetros do RAG simples:
- `chunk_size=4_000`: Chunks grandes para garantir contexto rico (o reranker vai filtrar, então podemos começar com mais)
- `chunk_overlap=20`: Pequena sobreposição para não perder contexto na divisão
- `add_start_index=True`: Rastreia a posição original de cada chunk no documento

In [12]:
# SEPARAR EM CHUNKS

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4_000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

### Vector Store (Naive Retriever Base)

- `Chroma` armazena os embeddings dos chunks para busca por similaridade
- `persist_directory='naiveDB'`: Diretório separado dos outros notebooks (cada RAG tem seu próprio vector store)
- Este vectorstore será usado como **retriever base** — a primeira etapa da busca, ainda sem reranking

In [13]:
# SALVAR OS CHUNKS NO VECTOR DB

vectordb = Chroma(embedding_function=embeddings_model, persist_directory='naiveDB')

### Naive Retriever — Primeira Etapa da Busca

- `search_kwargs={'k': 10}`: Recupera os **10 chunks** mais similares por embedding
- "Naive" porque usa apenas similaridade vetorial — sem entender contexto ou relevância real
- **Por que k=10?** Propositalmente amplo: o reranker vai receber esses 10 e selecionar apenas os 3 melhores
- Quanto mais candidatos para o reranker, melhor a qualidade final — mas maior o custo de processamento

In [ ]:
# CARREGAR O DB

naive_retreiver = vectordb.as_retriever(search_kwargs={ 'k': 10 })

### Reranking — Segunda Etapa da Busca

Esta é a parte central e diferencial deste notebook.

**Por que reranking?**
Embeddings medem similaridade semântica superficial (palavras/temas parecidos), mas não medem *relevância real* para responder uma pergunta específica. O reranker usa um modelo de linguagem dedicado que entende a relação entre pergunta e documento com muito mais profundidade.

**CohereRerank**:
- `model='rerank-multilingual-v3.0'`: Modelo multilingue (funciona bem em português)
- `top_n=3`: Dos 10 candidatos do naive retriever, seleciona apenas os **3 mais relevantes**
- Internamente, o modelo analisa cada par (pergunta, chunk) e atribui um score de relevância
- Reordena e filtra os documentos por esse score

**ContextualCompressionRetriever**:
- Orquestra os dois estágios: `base_retriever` (naive, k=10) → `base_compressor` (rerank, top_n=3)
- O nome "compressão" vem da ideia de comprimir 10 resultados em 3 mais relevantes
- Resultado final: 3 chunks altamente relevantes enviados ao LLM

In [19]:
rerank = CohereRerank(model='rerank-multilingual-v3.0', top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retreiver
)

### Prompt para Respostas sobre Legislação

Mesmo padrão do `parent-rag.ipynb` com `from_template`:
- `{question}`: Placeholder para a pergunta (note: aqui usa `question`, não `input` ou `query`)
- `{context}`: Placeholder para os 3 chunks rerankeados
- O papel de "especialista em legislação e tecnologia" direciona o LLM ao domínio do documento

In [23]:
TEMPLATE = """
Você é um especialista em legislação e tecnologia. Responda a pergunta abaixo utilizando o contexto informado.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

### Chain RAG com Reranking (LCEL)

Monta o pipeline completo em dois blocos e encadeia com `|`:

**RunnableParallel** (`setup_retrival`):
- `'question': RunnablePassthrough()` → Passa a pergunta direto para o prompt
- `'context': compressor_retriever` → Executa os dois estágios (naive k=10 → rerank top_n=3)
- Retorna `{'question': '...', 'context': [chunk1, chunk2, chunk3]}`

**Chain final**:
```
setup_retrival | rag_prompt | llm | output_parser
```
- `rag_prompt`: Formata pergunta + contexto rerankeado no template
- `llm`: Gera a resposta com base nos 3 chunks mais relevantes
- `output_parser`: Extrai a string da resposta do `AIMessage`

In [24]:
setup_retrival = RunnableParallel({
    'question': RunnablePassthrough(),
    'context': compressor_retriever,
})

output_parser = StrOutputParser()

compressor_retriever_chain = setup_retrival | rag_prompt | llm | output_parser

### Executando o Pipeline Rerank RAG

O fluxo completo ao invocar com uma pergunta:
1. Pergunta entra no `RunnableParallel`
2. `naive_retriever` busca 10 chunks por similaridade de embedding no Chroma
3. `CohereRerank` recebe os 10 chunks + pergunta, analisa cada par e seleciona os 3 mais relevantes
4. Os 3 chunks rerankeados + pergunta preenchem o template
5. LLM gera a resposta com contexto altamente filtrado
6. `StrOutputParser` retorna a resposta como string

In [25]:
compressor_retriever_chain.invoke('Quais são os principais pontos de risco do marco legal de IA')

'Os principais pontos de risco do marco legal de IA estão relacionados à privacidade dos dados, viés algorítmico, transparência e responsabilidade. A falta de regras claras e consistentes pode resultar em uso inadequado de informações pessoais, discriminação injusta e dificuldade em responsabilizar os agentes responsáveis por danos causados pelo uso de sistemas de IA. É importante que a legislação estabeleça diretrizes sólidas para mitigar esses riscos e garantir a confiança dos usuários na tecnologia de IA.'